In [ ]:
!apt-get install -y cmake build-essential

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
build-essential is already the newest version (12.9ubuntu3).
cmake is already the newest version (3.22.1-1ubuntu1.22.04.2).
0 upgraded, 0 newly installed, 0 to remove and 57 not upgraded.


In [ ]:
!git clone https://github.com/ggerganov/llama.cpp

Cloning into 'llama.cpp'...
remote: Enumerating objects: 122710, done.
remote: Counting objects: 100% (872/872), done.
remote: Compressing objects: 100% (255/255), done.
remote: Total 122710 (delta 728), reused 618 (delta 617), pack-reused 121838 (from 3)
Receiving objects: 100% (122710/122710), 428.21 MiB | 29.80 MiB/s, done.
Resolving deltas: 100% (86674/86674), done.


In [ ]:
%cd llama.cpp

/content/llama.cpp


In [ ]:
!ls -lh

total 408K
-rw-r--r--  1 root root  12K Sep  5 21:48 AGENTS.md
drwxr-xr-x  2 root root 4.0K Sep  5 21:48 app
-rw-r--r--  1 root root  88K Sep  5 21:48 AUTHORS
drwxr-xr-x  5 root root 4.0K Sep  5 21:48 benches
-rwxr-xr-x  1 root root  25K Sep  5 21:48 build-xcframework.sh
drwxr-xr-x  2 root root 4.0K Sep  5 21:48 ci
-rw-r--r--  1 root root  106 Sep  5 21:48 CLAUDE.md
drwxr-xr-x  2 root root 4.0K Sep  5 21:48 cmake
-rw-r--r--  1 root root  11K Sep  5 21:48 CMakeLists.txt
-rw-r--r--  1 root root 4.5K Sep  5 21:48 CMakePresets.json
-rw-r--r--  1 root root 6.4K Sep  5 21:48 CODEOWNERS
drwxr-xr-x  3 root root 4.0K Sep  5 21:48 common
-rw-r--r--  1 root root  13K Sep  5 21:48 CONTRIBUTING.md
drwxr-xr-x  2 root root 4.0K Sep  5 21:48 conversion
-rwxr-xr-x  1 root root  13K Sep  5 21:48 convert_hf_to_gguf.py
-rwxr-xr-x  1 root root  28K Sep  5 21:48 convert_hf_to_gguf_update.py
-rwxr-xr-x  1 root root  19K Sep  5 21:48 convert_llama_ggml_to_gguf.py
-rwxr-xr-x  1 root root  23K Sep  5 21:48 conv

In [ ]:
# The Makefile build was removed upstream - llama.cpp is CMake-only now.
# On a GPU runtime add -DGGML_CUDA=ON to the configure line (slower build, much faster inference).
!cmake -B build -DCMAKE_BUILD_TYPE=Release
!cmake --build build --config Release -j $(nproc)

-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- llama.cpp version: 0.4.0-dev
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Found Ope

In [ ]:
from huggingface_hub import snapshot_download

model_path = snapshot_download(
    repo_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    local_dir="tinyllama-hf",
    local_dir_use_symlinks=False
)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

In [ ]:
!pip install mistral-common

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 99.1 MB/s eta 0:00:00


In [ ]:
!python3 convert_hf_to_gguf.py ./tinyllama-hf \
    --outfile ./tinyllama-1.1b-chat.gguf

INFO:hf-to-gguf:Loading model: tinyllama-hf
INFO:numexpr.utils:NumExpr defaulting to 2 threads.
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:hf-to-gguf:heuristics detected bfloat16 tensor dtype, setting --outtype bf16
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,               torch.bfloat16 --> BF16, shape = {2048, 32000}
INFO:hf-to-gguf:token_embd.weight,           torch.bfloat16 --> BF16, shape = {2048, 32000}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.bfloat16 --> BF16, shape = {5632, 2048}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.bfloat16 --> BF16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.bfloat16 --> BF16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.bfloat16 --

In [ ]:
!./build/bin/llama-quantize ./tinyllama-1.1b-chat.gguf ./tinyllama-1.1b-chat-q4_0.gguf Q4_0

version: 0.4.0-dev (build 10820, commit 74a7c897f)
built with GNU 11.4.0 for Linux x86_64
llama_quantize: quantizing './tinyllama-1.1b-chat.gguf' to './tinyllama-1.1b-chat-q4_0.gguf' as Q4_0
llama_model_loader: loaded meta data with 45 key-value pairs and 201 tensors from ./tinyllama-1.1b-chat.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Tinyllama Hf
llama_model_loader: - kv   3:                         general.size_label str              = 1.1B
llama_model_loader: - kv   4:                            general.license str              = apache-2.0
llama_model_loader: - kv   5:                      general.dataset.count u

In [ ]:
!./build/bin/llama-cli -m ./tinyllama-1.1b-chat.gguf -p "Explain quantization in LLMs" -n 100



Loading model... |-\|/-\|/- 

▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b10820-74a7c897f
model      : ./tinyllama-1.1b-chat.gguf
ftype      : BF16
modalities : text

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat history
  /read <file>        add a text file
  /glob <pattern>     add text files using globbing pattern



> Explain quantization in LLMs
Quantization is a technique used in machine learning models to reduce the range of values used for training data. This technique is used to reduce the size of the representation space in the neural network, thereby reducing the size of the weights and the complexity of the model.

In Lattice-based Lightning Models (LLM

In [ ]:
import os
os.getcwd()

'/content/llama.cpp'

In [ ]:
!ls build/bin | head -20   # llama-cli, llama-quantize, llama-server, ...

In [ ]:
# %cd build   # not needed: `cmake -B build` above keeps us at the repo root

`cmake -B build` -> generates the build system (a Makefile) inside `build/`.
Think of it as creating a blueprint for how the code should be compiled.

`cmake --build build` -> follows those instructions and compiles the binaries.
This is the actual construction step - turning source code into executables.

Output = `build/bin/llama-cli`, `build/bin/llama-quantize`, etc. -> now you can run them.
(The old names `./main` and `./quantize` were renamed upstream in 2024.)

In [ ]:
# !cmake ..   # superseded by `cmake -B build` in the build cell above

In [ ]:
# !make       # superseded by `cmake --build build` in the build cell above

cmake + make -> build the inference engine (`build/bin/llama-cli`).

`llama-quantize` -> prepare a quantized .gguf from the F16 .gguf.

`build/bin/llama-cli` -> run inference with your prompt.

In [ ]:
!ls

AGENTS.md		       include
app			       LICENSE
AUTHORS			       licenses
benches			       Makefile
build			       media
build-xcframework.sh	       models
ci			       mypy.ini
CLAUDE.md		       pocs
cmake			       pyproject.toml
CMakeLists.txt		       pyrightconfig.json
CMakePresets.json	       README.md
CODEOWNERS		       requirements
common			       requirements.txt
CONTRIBUTING.md		       scripts
conversion		       SECURITY.md
convert_hf_to_gguf.py	       skills
convert_hf_to_gguf_update.py   src
convert_llama_ggml_to_gguf.py  tests
convert_lora_to_gguf.py        tinyllama-1.1b-chat.gguf
docs			       tinyllama-1.1b-chat-q4_0.gguf
examples		       tinyllama-hf
flake.nix		       tools
ggml			       ty.toml
gguf-py			       vendor
grammars


In [ ]:
os.getcwd()

'/content/llama.cpp'

In [ ]:
!ls -lh /content/llama.cpp/

total 2.7G
-rw-r--r--  1 root root  12K Sep  5 21:48 AGENTS.md
drwxr-xr-x  2 root root 4.0K Sep  5 21:48 app
-rw-r--r--  1 root root  88K Sep  5 21:48 AUTHORS
drwxr-xr-x  5 root root 4.0K Sep  5 21:48 benches
drwxr-xr-x 14 root root 4.0K Sep  5 21:48 build
-rwxr-xr-x  1 root root  25K Sep  5 21:48 build-xcframework.sh
drwxr-xr-x  2 root root 4.0K Sep  5 21:48 ci
-rw-r--r--  1 root root  106 Sep  5 21:48 CLAUDE.md
drwxr-xr-x  2 root root 4.0K Sep  5 21:48 cmake
-rw-r--r--  1 root root  11K Sep  5 21:48 CMakeLists.txt
-rw-r--r--  1 root root 4.5K Sep  5 21:48 CMakePresets.json
-rw-r--r--  1 root root 6.4K Sep  5 21:48 CODEOWNERS
drwxr-xr-x  3 root root 4.0K Sep  5 21:48 common
-rw-r--r--  1 root root  13K Sep  5 21:48 CONTRIBUTING.md
drwxr-xr-x  3 root root 4.0K Sep  5 22:05 conversion
-rwxr-xr-x  1 root root  13K Sep  5 21:48 convert_hf_to_gguf.py
-rwxr-xr-x  1 root root  28K Sep  5 21:48 convert_hf_to_gguf_update.py
-rwxr-xr-x  1 root root  19K Sep  5 21:48 convert_llama_ggml_to_gguf.p

In [ ]:
!./build/bin/llama-cli -m ./tinyllama-1.1b-chat.gguf -p "What is quantization in LLMs?" -n 100 -no-cnv

✅ llama-cli (official main runner)

✅ llama-run (multi-prompt / batch)

In [ ]:
import os
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings

# ==== Step 1: Load your document ====
def load_documents(file_path):
    print(f"Loading file: {file_path}")
    loader = TextLoader(file_path)
    return loader.load()

# ==== Step 2: Chunk the text ====
def chunk_documents(documents, chunk_size=500, chunk_overlap=100):
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    return splitter.split_documents(documents)

# ==== Step 3: Create or load FAISS VectorStore ====
def create_or_load_faiss(chunks, embedding_model, index_path="faiss_index"):
    if os.path.exists(index_path):
        print("Loading existing FAISS index...")
        return FAISS.load_local(index_path, embedding_model)
    print("Creating new FAISS index...")
    db = FAISS.from_documents(chunks, embedding_model)
    db.save_local(index_path)
    return db

# ==== Step 4: Retrieve relevant chunks ====
def get_context_from_query(query, retriever):
    docs = retriever.get_relevant_documents(query)
    return "\n\n".join([doc.page_content for doc in docs])

# ==== Step 5: Build prompt & write to file ====
def build_prompt_file(context, query, prompt_file="prompt.txt"):
    prompt = f"""[INST] <<SYS>>
You are a helpful AI assistant. Use the context to answer the question.
<</SYS>>

Context:
{context}

Question: {query}
Answer: [/INST]
"""
    with open(prompt_file, "w") as f:
        f.write(prompt)
    print(f"Prompt written to {prompt_file}")

# ==== Step 6: Run llama.cpp with GGUF ====
def run_llama_cli(gguf_path, prompt_file="prompt.txt", n_predict=200):
    print("Running inference with llama.cpp...")
    os.system(f"./build/bin/llama-cli -m {gguf_path} -f {prompt_file} --n-predict {n_predict} -no-cnv")

# ==== === MAIN PIPELINE === ===
def rag_pipeline(
    doc_path="my_notes.txt",
    gguf_model_path="./tinyllama-1.1b-chat-q4_0.gguf",
    user_query="What is quantization in LLMs?"
):
    # Load & split
    docs = load_documents(doc_path)
    chunks = chunk_documents(docs)

    # Embeddings
    embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

    # FAISS
    vectorstore = create_or_load_faiss(chunks, embedding_model)
    retriever = vectorstore.as_retriever()

    # RAG
    context = get_context_from_query(user_query, retriever)
    build_prompt_file(context, user_query)

    # Inference
    run_llama_cli(gguf_model_path)

# ==== Entry Point ====
if __name__ == "__main__":
    rag_pipeline()


GGML (Georgi Gerganov Machine Learning) ek C-based runtime + tensor library hai jo:

Low-level CPU/GPU optimized inference engine hai

Mainly llama.cpp, whisper.cpp, stable-diffusion.cpp jaise projects use karte hain

Original format tha before GGUF came in

No Python dependency – pure C/C++ based

📌 GGUF = file format

📌 GGML = inference engine + tensor library

Practical: Run a GGML-Format LLM Model (like ggml-model-q4.bin)

🧰 Tools:

✅ llama.cpp (same as GGUF)

✅ Prequantized GGML model (e.g., from TheBloke)

✅ main binary from llama.cpp

Step-by-Step GGML Inference

🔹 Step 1: Clone & Build llama.cpp

In [ ]:
# Already cloned and built at the top of this notebook - nothing to do here.
# Note: `!cd llama.cpp` does NOT persist (each ! runs in its own shell) - use %cd.
# And `!make` fails: the Makefile build was removed upstream.
!ls /content/llama.cpp/build/bin | head -20

Step 2: Download a GGML Model

Use any of the prequantized .bin models:

In [ ]:
# llama.cpp dropped GGML (.bin) support entirely - it is GGUF-only now,
# so the old TheBloke *-GGML repos cannot be run any more. Pull a GGUF instead:
!wget -q --show-progress https://huggingface.co/TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF/resolve/main/tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf -O /content/tinyllama-q4.gguf

Step 3: Run Inference

In [ ]:
!/content/llama.cpp/build/bin/llama-cli -m /content/tinyllama-q4.gguf -p "What is quantization in machine learning?" -n 100 -no-cnv

Prompt: What is quantization in machine learning?

Output: Quantization is the process of reducing the precision of the weights and activations of a neural network. It is commonly used for...

In [ ]:
!pip -q install llama-cpp-python

from llama_cpp import Llama
# Uses the GGUF downloaded above; set n_gpu_layers=0 for CPU-only runtimes.
llm = Llama(model_path="/content/tinyllama-q4.gguf", n_gpu_layers=35)
out = llm("Explain GPTQ vs AWQ in 2 lines.", max_tokens=80)
print(out["choices"][0]["text"])